# Setup

In [1]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from thesis_assyrian_relief.utils.config import load_yaml_config
from sklearn.model_selection import train_test_split



def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    return here

paths_yaml = _repo_root() / "configs" / "style_dinov2.yaml"
config = load_yaml_config(paths_yaml)

dataset_xlsx = config["data"]["dataset_xlsx"]
image_root = Path(config["data"]["image_root"])

In [3]:
df_full = pd.read_excel(dataset_xlsx)

df_full["Authority"] = df_full["Authority"].str.strip().str.replace('\xa0', '', regex=False)

df_full.head()

,URL,Relief_ID,Tags,Museum,Authority,Grayscale,style_analysis_use,background_masking,Tier,Code,...,facade,Panels,Materials,Weight,Height,Width,Thickness,Curators_Comments,Inscription_translation,Unnamed: 30
0,https://www.metmuseum.org/art/collection/searc...,MET 31.72.2,NaN,Metropolitan,Ashurnasirpal II,NaN,core_train,NaN,B,NaN,...,NaN,NaN,Gypsum alabaster,NaN,233.7 cm,167.6 cm,6.4 cm,NaN,NaN,NaN
1,https://www.britishmuseum.org/collection/objec...,BM 118810,Head,BM,Sargon II,1.0,core_train,1.0,A,NaN,...,NaN,NaN,limestone,NaN,8 inch,NaN,NaN,NaN,NaN,NaN
2,https://www.britishmuseum.org/collection/objec...,BM 118811,Head,BM,Sargon II,1.0,extended_train,1.0,B,NaN,...,NaN,NaN,stone,NaN,25 inch,20 inch,NaN,NaN,NaN,NaN
3,https://www.britishmuseum.org/collection/objec...,BM 118820,Head,BM,Sargon II,1.0,extended_train,1.0,A,NaN,...,NaN,NaN,stone,NaN,2 feet,19 inch,NaN,NaN,NaN,NaN
4,https://www.britishmuseum.org/collection/objec...,BM 118822,king/queen\nofficial\nprince/princess (?)\narm...,BM,Sargon II,0.0,core_train,1.0,B,NaN,...,NaN,NaN,gypsum,NaN,290 cm,230 cm,NaN,A duplicate of this scene was found in the pal...,NaN,NaN


In [7]:
df_full["Museum_id"].value_counts()

Museum_id
AO 19901     2
AO 19902     2
AO 19903     2
AO 19904     2
AO 2254      2
            ..
HM 3943      1
HM 3944      1
HM 3945      1
S.856.3.1    1
DIA 50.32    1
Name: count, Length: 525, dtype: int64

# pie chart Museum Distribution

In [15]:
museum_counts = (
    df_full.dropna(subset=["Museum"])
    .groupby("Museum")["Relief_ID"]
    .nunique()
    .sort_values(ascending=False)
)

top_n = 4
top_museums = museum_counts.head(top_n)
other_total = int(museum_counts.iloc[top_n:].sum())

pie_labels = list(top_museums.index) + ["Other"]
pie_values = list(top_museums.values) + [other_total]

fig = go.Figure(
    data=[
        go.Pie(
            labels=pie_labels,
            values=pie_values,
            textinfo="label+percent",
            texttemplate="%{label}<br>%{percent} (%{value})",
            hovertemplate="%{label}: %{value} unique reliefs (%{percent})<extra></extra>",
            sort=False,
        )
    ],
    layout=go.Layout(height=600),
)

fig.update_layout(
    title="<b>Distribution of unique reliefs by museum (top 4 + Other)</b>",
    title_xanchor="center",
    title_x=0.5,
    template="plotly_white",
    legend=dict(font=dict(size=14)),
)
fig.update_traces(textfont_size=20)
fig.show()

In [31]:
origin_name = "Neo-Assyrian Empire"
origin_lat, origin_lon = 36.3590, 43.1530

labels_top_n = 4
labels_include =  ["Pergamon Museum", "Hermitage"]
labels_exclude = []

museum_locations = {
    "bm": ("British Museum (London)", 51.5194, -0.1270),
    "british museum": ("British Museum (London)", 51.5194, -0.1270),
    "met": ("Metropolitan Museum (New York)", 40.7794, -73.9632),
    "metropolitan": ("Metropolitan Museum (New York)", 40.7794, -73.9632),
    "metropolitan museum of art": ("Metropolitan Museum (New York)", 40.7794, -73.9632),
    "louvre": ("Louvre (Paris)", 48.8606, 2.3376),
    "musee du louvre": ("Louvre (Paris)", 48.8606, 2.3376),
    "vorderasiatisches museum": ("Vorderasiatisches Museum (Berlin)", 52.5212, 13.3964),
    "vam": ("Vorderasiatisches Museum (Berlin)", 52.5212, 13.3964),
    "pergamon": ("Pergamon Museum (Berlin)", 52.5212, 13.3964),
    "pergamonmuseum": ("Pergamon Museum (Berlin)", 52.5212, 13.3964),
    "iraq museum": ("Iraq Museum (Baghdad)", 33.3286, 44.3879),
    "national museum of iraq": ("Iraq Museum (Baghdad)", 33.3286, 44.3879),
    "mosul museum": ("Mosul Museum", 36.3489, 43.1577),
    "yale": ("Yale University Art Gallery (New Haven)", 41.3083, -72.9311),
    "yale university art gallery": ("Yale University Art Gallery (New Haven)", 41.3083, -72.9311),
    "yale babylonian collection": ("Yale University Art Gallery (New Haven)", 41.3083, -72.9311),
    "mfa": ("Museum of Fine Arts (Boston)", 42.3394, -71.0941),
    "boston mfa": ("Museum of Fine Arts (Boston)", 42.3394, -71.0941),
    "museum of fine arts": ("Museum of Fine Arts (Boston)", 42.3394, -71.0941),
    "detroit institute of arts": ("Detroit Institute of Arts", 42.3594, -83.0644),
    "dia": ("Detroit Institute of Arts", 42.3594, -83.0644),
    "brooklyn museum": ("Brooklyn Museum", 40.6713, -73.9636),
    "cincinnati art museum": ("Cincinnati Art Museum", 39.1141, -84.4983),
    "hermitage": ("Hermitage (St. Petersburg)", 59.9398, 30.3146),
    "walters art museum": ("Walters Art Museum (Baltimore)", 39.2961, -76.6166),
    "penn museum": ("Penn Museum (Philadelphia)", 39.9492, -75.1916),
    "university of pennsylvania": ("Penn Museum (Philadelphia)", 39.9492, -75.1916),
    "israel museum": ("Israel Museum (Jerusalem)", 31.7726, 35.2042),
    "blmj": ("Bible Lands Museum (Jerusalem)", 31.7889, 35.2076),
    "bible lands museum": ("Bible Lands Museum (Jerusalem)", 31.7889, 35.2076),
    "rockefeller": ("Rockefeller Archaeological Museum (Jerusalem)", 31.7855, 35.2378),
    "vatican": ("Vatican Museums (Rome)", 41.9065, 12.4536),
    "vatican museums": ("Vatican Museums (Rome)", 41.9065, 12.4536),
    "barracco": ("Museo Barracco (Rome)", 41.8959, 12.4717),
    "barraco": ("Museo Barracco (Rome)", 41.8959, 12.4717),
    "museo archeologico": ("Museo Archeologico (Florence)", 43.7747, 11.2618),
    "rmah": ("Royal Museums of Art and History (Brussels)", 50.8395, 4.3927),
    "brussels": ("Royal Museums of Art and History (Brussels)", 50.8395, 4.3927),
    "rom": ("Royal Ontario Museum (Toronto)", 43.6677, -79.3947),
    "royal ontario museum": ("Royal Ontario Museum (Toronto)", 43.6677, -79.3947),
    "ngc": ("National Gallery of Canada (Ottawa)", 45.4296, -75.6989),
    "national gallery of canada": ("National Gallery of Canada (Ottawa)", 45.4296, -75.6989),
    "rah": ("Real Academia de la Historia (Madrid)", 40.4153, -3.6996),
    "real academia de la historia": ("Real Academia de la Historia (Madrid)", 40.4153, -3.6996),
    "hma": ("Hood Museum of Art (Hanover, NH)", 43.7036, -72.2887),
    "hood museum of art": ("Hood Museum of Art (Hanover, NH)", 43.7036, -72.2887),
    "oriental institute": ("Oriental Institute (Chicago)", 41.7886, -87.5970),
    "isac": ("Oriental Institute (Chicago)", 41.7886, -87.5970),
    "art institute of chicago": ("Art Institute of Chicago", 41.8796, -87.6237),
    "ashmolean": ("Ashmolean Museum (Oxford)", 51.7553, -1.2603),
    "fitzwilliam": ("Fitzwilliam Museum (Cambridge)", 52.2000, 0.1196),
    "bowdoin": ("Bowdoin College Museum of Art", 43.9072, -69.9628),
    "dartmouth": ("Hood Museum of Art (Hanover, NH)", 43.7036, -72.2887),
    "williams": ("Williams College Museum of Art", 42.7117, -73.2034),
    "amherst": ("Mead Art Museum (Amherst)", 42.3711, -72.5152),
    "mount holyoke": ("Mount Holyoke College Art Museum", 42.2553, -72.5752),
    "smith": ("Smith College Museum of Art", 42.3175, -72.6394),
    "vassar": ("Vassar College Loeb Art Center", 41.6873, -73.8957),
    "middlebury": ("Middlebury College Museum of Art", 44.0086, -73.1745),
    "princeton": ("Princeton University Art Museum", 40.3464, -74.6571),
    "harvard": ("Harvard Art Museums", 42.3744, -71.1144),
    "smithsonian": ("Smithsonian (Washington, D.C.)", 38.8881, -77.0260),
    "freer": ("Freer Gallery of Art (Washington, D.C.)", 38.8881, -77.0260),
    "national museum of scotland": ("National Museum of Scotland (Edinburgh)", 55.9469, -3.1903),
    "burrell collection": ("Burrell Collection (Glasgow)", 55.8244, -4.3133),
    "burrell": ("Burrell Collection (Glasgow)", 55.8244, -4.3133),
    "world museum liverpool": ("World Museum (Liverpool)", 53.4101, -2.9817),
    "birmingham museum": ("Birmingham Museum and Art Gallery", 52.4796, -1.9036),
    "manchester museum": ("Manchester Museum", 53.4664, -2.2336),
}

museum_counts = (
    df_full.dropna(subset=["Museum"])
    .groupby("Museum")["Relief_ID"]
    .nunique()
    .sort_values(ascending=False)
)

records, unmapped = [], []
for museum, count in museum_counts.items():
    key = str(museum).strip().lower()
    info = museum_locations.get(key)
    if info is None:
        unmapped.append((museum, int(count)))
        continue
    label, lat, lon = info
    records.append({"museum": museum, "label": label, "count": int(count), "lat": lat, "lon": lon})

if unmapped:
    print("Museums without mapped coordinates (skipped) -- add them to `museum_locations`:")
    for m, c in unmapped:
        print(f"  - {m!r}: {c} unique reliefs")

map_df = pd.DataFrame(records).sort_values("count", ascending=False).reset_index(drop=True)
max_count = int(map_df["count"].max()) if not map_df.empty else 1

_include_lower = {s.strip().lower() for s in labels_include}
_exclude_lower = {s.strip().lower() for s in labels_exclude}
_top_n_idx = set(map_df.head(labels_top_n).index) if labels_top_n else set()

def _should_label(idx, row):
    name_lower = str(row["label"]).lower()
    museum_lower = str(row["museum"]).lower()
    if name_lower in _exclude_lower or museum_lower in _exclude_lower:
        return False
    if idx in _top_n_idx:
        return True
    return name_lower in _include_lower or museum_lower in _include_lower

display_text = [
    row["label"] if _should_label(i, row) else ""
    for i, row in map_df.iterrows()
]

from collections import defaultdict
cluster_bucket = 1.5
buckets = defaultdict(list)
for i, row in map_df.iterrows():
    buckets[(round(row["lat"] / cluster_bucket), round(row["lon"] / cluster_bucket))].append(i)

position_cycle = ["top center", "bottom center", "top right", "bottom left", "top left", "bottom right"]
text_positions = ["top center"] * len(map_df)
for indices in buckets.values():
    if len(indices) > 1:
        indices_sorted = sorted(indices, key=lambda i: -int(map_df.loc[i, "count"]))
        for k, idx in enumerate(indices_sorted):
            text_positions[idx] = position_cycle[k % len(position_cycle)]

fig = go.Figure()

for _, row in map_df.iterrows():
    fig.add_trace(
        go.Scattergeo(
            lon=[origin_lon, row["lon"]],
            lat=[origin_lat, row["lat"]],
            mode="lines",
            line=dict(width=1 + 4 * (row["count"] / max_count), color="rgba(200, 30, 30, 0.55)"),
            showlegend=False,
            hoverinfo="skip",
        )
    )

fig.add_trace(
    go.Scattergeo(
        lon=map_df["lon"],
        lat=map_df["lat"],
        text=display_text,
        customdata=[f"{r.label}<br>{r.count} unique reliefs" for r in map_df.itertuples()],
        mode="markers+text",
        textposition=text_positions,
        textfont=dict(size=11, color="black"),
        marker=dict(
            size=8 + 28 * (map_df["count"] / max_count),
            color=map_df["count"],
            colorscale="Reds",
            line=dict(width=1, color="black"),
            colorbar=dict(title="Unique<br>reliefs"),
        ),
        name="Museums",
        hovertemplate="%{customdata}<extra></extra>",
    )
)

fig.add_trace(
    go.Scattergeo(
        lon=[origin_lon],
        lat=[origin_lat],
        text=[origin_name],
        mode="markers+text",
        marker=dict(size=16, color="gold", symbol="star", line=dict(width=1.5, color="black")),
        textposition="bottom center",
        textfont=dict(size=12, color="black"),
        name=origin_name,
        hoverinfo="text",
    )
)

fig.update_layout(
    title="<b>Dispersion of Assyrian reliefs from Nineveh to modern museums</b>",
    title_x=0.5,
    geo=dict(
        scope="world",
        projection_type="natural earth",
        showland=True,
        landcolor="rgb(243, 243, 243)",
        countrycolor="rgb(204, 204, 204)",
        showcoastlines=True,
        coastlinecolor="rgb(150, 150, 150)",
        showocean=True,
        oceancolor="rgb(230, 240, 255)",
    ),
    height=700,
    template="plotly_white",
    legend=dict(font=dict(size=12)),
)
fig.show()

# Unique reliefs vs core_train reliefs

In [23]:
import plotly.graph_objects as go

monarch_colors = {
    "Ashurnasirpal II": "blue",
    "Ashurbanipal": "red",
    "Sargon II": "green",
    "Sennacherib": "purple",
    "Tiglath-Pileser III": "orange",
}
monarchs = list(monarch_colors.keys())
test_category = "core_train"

df_monarchs = df_full[df_full["Authority"].isin(monarchs)]

total_unique = (
    df_monarchs.groupby("Authority")["Relief_ID"]
    .nunique()
    .reindex(monarchs, fill_value=0)
)
test_unique = (
    df_monarchs[df_monarchs["style_analysis_use"] == test_category]
    .groupby("Authority")["Relief_ID"]
    .nunique()
    .reindex(monarchs, fill_value=0)
)

bar_colors = [monarch_colors[m] for m in monarchs]

fig = go.Figure(layout=go.Layout(height=800))
fig.add_trace(
    go.Bar(
        name="Total unique",
        x=monarchs,
        y=total_unique.values,
        marker_color=bar_colors,
        text=total_unique.values,
        textposition="outside",
        textfont=dict(size=16)
    )
)
fig.add_trace(
    go.Bar(
        name=f"Unique in {test_category}",
        x=monarchs,
        y=test_unique.values,
        marker_color=bar_colors,
        marker_pattern_shape="/",
        text=test_unique.values,
        textposition="outside",
    )
)

fig.update_layout(
    title="<b>Unique reliefs per monarch: total vs. refined dataet</b>",
    title_xanchor='center',
    title_x=0.5,
    xaxis_title="Monarch",
    yaxis_title="Unique reliefs",
    barmode="group",
    template="plotly_white",
    legend_title_text="",
    uniformtext_minsize=16, 
    uniformtext_mode='show',
     legend=dict(
        font=dict(
            size=20  # Set your desired font size here
        )
     ),
    
)
fig.update_xaxes(title_font_size=24, tickfont=dict(size=18))
fig.update_yaxes(title_font_size=24)
fig.show()